WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

In [ ]:
import dataclasses

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polyscope as ps
import trimesh

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.high_res_plot_settings)

### Rod

In [ ]:
rod_length = 0.15
num_elems = 90
num_nodes = num_elems + 1
shear_modulus = 1.0e6

fiber_offset = 0.005
fiber_radius = 0.003
offsets_local = np.array(
  [[fiber_offset, 0.0], [-fiber_offset, 0.0],
   [0.0, fiber_offset], [0.0, -fiber_offset]]
)
num_fibers = 4
area_per_fiber = np.pi * fiber_radius**2

# A deterministic half-wave bow breaks the symmetry of the threading problem, which
# would otherwise sit on a bifurcation the moment the tip meets the fillet. The
# amplitude is small enough that the reference is effectively straight.
initial_bow_ratio = 1.0e-3

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda s: jnp.array([s, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, rod_length, num_nodes),
  fiber_offsets=jnp.asarray(np.tile(offsets_local, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), area_per_fiber),
  fiber_mass_density=jnp.ones((num_elems, num_fibers)),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
  initial_bow_ratio=initial_bow_ratio,
)
contact_radii = np.asarray(rod.contact_radii)
bending_stiffness = (
  3.0 * shear_modulus * np.sum(area_per_fiber * offsets_local[:, 0] ** 2)
)
print(f"contact radius = {contact_radii.max():.4f} m")
print(f"EI             = {bending_stiffness:.4e} N m^2")

### Obstacle

In [ ]:
bore_radius = 0.020
fillet_radius = 0.060
inlet_length = 0.150

pipe = trimesh.load_mesh("crops/data/artery/0011_H_AO_H/0011_H_AO_H_artery_wall.stl")
if isinstance(pipe, trimesh.Scene):
  pipe = trimesh.util.concatenate(pipe.dump())
print(f"watertight {pipe.is_watertight}, extents {np.round(pipe.extents, 4)} m")
print(f"clearance = {bore_radius - contact_radii.max():.4f} m")

In [ ]:
# Padding at least one contact radius puts the grid boundary where the contact force
# is already identically zero, so the discontinuity across it is invisible to the
# solver. Resolution is set by the bore, not by the overall size of the mesh.
padding = 3.0 * contact_radii.max()
grid_shape = (96, 96, 40)
wall_penalty_stiffness = 1.0e6

obstacle = _contact.Obstacle.from_mesh(
  mesh=pipe,
  grid_shape=grid_shape,
  wall_penalty_stiffness=wall_penalty_stiffness,
  padding=padding,
  extrap_fill_value=10.0,
)
spacing = np.asarray(obstacle.spacing)
print(f"grid spacing {np.round(spacing, 5)} m")
print(f"cells across the bore = {2.0 * bore_radius / spacing.max():.1f}  (want > 10)")

In [ ]:
# The pipe mouth sits at the origin of its own body frame, so translating that frame
# along x positions the mouth in the lab. Starting just beyond the free tip and
# advancing toward the clamp draws the pipe back over the rod, threading the tip
# deeper with every step.
num_steps = 50
mouth_start = rod_length + 0.02
mouth_end = -0.05
mouth_positions = np.linspace(mouth_start, mouth_end, num_steps)

movement = _contact.ObstacleMovement.from_translations(
  jnp.stack(
    [jnp.asarray(mouth_positions), jnp.zeros(num_steps), jnp.zeros(num_steps)],
    axis=-1,
  )
)
obstacle = dataclasses.replace(obstacle, movement=movement)
print(f"mouth travels {mouth_start:.3f} -> {mouth_end:.3f} m over {num_steps} steps")
print(f"deepest insertion = {rod_length - mouth_end:.3f} m along the pipe")

In [ ]:
# Sanity check the pose map before solving. A point on the rod axis should read a
# large positive distance while the pipe is still ahead of it, and a small positive
# distance once the pipe has slid over it, since the lumen is free space.
probe = jnp.array([[0.5 * rod_length, 0.0, 0.0]])
for step in [0, num_steps // 2, num_steps - 1]:
  obj_trans, obj_rot_mat = obstacle.pose_at_step(step)
  phi = float(obstacle.compute_sdf(probe, obj_trans, obj_rot_mat)[0])
  print(f"step {step:3d}  mouth x = {float(obj_trans[0]):.4f}  phi = {phi:+.5f}  "
        f"overlap = {contact_radii.max() - phi:+.5f}")

### Boundary conditions and problem

In [ ]:
clamped_dofs = (
  _utils.Translation.X, _utils.Translation.Y, _utils.Translation.Z,
  _utils.Rotation.THETA_X, _utils.Rotation.THETA_Y, _utils.Rotation.THETA_Z,
)
bc = _bc.process_boundary_conditions(
  rod,
  [
    _bc.DirichletBC(
      values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
      node_targets=[np.array([0]) for _ in range(num_steps)],
      name=f"clamp_{dof}",
    )
    for dof in clamped_dofs
  ],
)
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 200, "threshold": 1e-6, "abs_threshold": 1e-6, "lam_min": 0.01,
    "line_search_max_iter": 15, "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6, "line_search_armijo_c": 1e-4,
  },
  linear_settings={"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-10},
)
problem = _fe.CosseratProblem(
  rod=rod, bc=bc, nr_solver=nr_solver, obstacle=obstacle
)

### Contact forces

`compute_contact_forces` belongs on `CosseratProblem`. It is defined here so the
notebook runs as-is; move it and delete this cell.

In [ ]:
def compute_contact_forces(problem, u: jnp.ndarray, bc_step_idx: int = 0):
  """Nodal contact force exerted by the obstacle on the rod.

  The quantity assembled inside the residual is the positive energy gradient, which
  acts as an internal force and is subtracted from the residual. The physical force
  applied to the rod is its negative, which is what is returned, so a positive
  component points in the direction the obstacle pushes.

  Only translational components are returned. The wall contact energy depends on the
  element centre position, which is the average of the two nodal translations, so
  the rotational components of the assembled vector are identically zero.

  Args:
    problem: `CosseratProblem` carrying the rod and the obstacle.
    u: Array of shape `(num_dofs,)` [m, rad] of nodal displacements and rotations.
    bc_step_idx: Load step index used to look up the obstacle pose.

  Returns:
    Array of shape `(num_nodes, num_translational_dofs_per_node)` [N].
  """
  obj_trans, obj_rot_mat = problem.obstacle.pose_at_step(bc_step_idx)
  f_contact, _ = _contact.compute_wall_contact_forces_and_tangent(
    u_elems=u[problem.rod.elem_dof_mat],
    rod=problem.rod,
    obstacle=problem.obstacle,
    compute_jacobian=False,
    obj_trans=obj_trans,
    obj_rot_mat=obj_rot_mat,
  )
  num_trans = problem.rod.num_translational_dofs_per_node
  return -f_contact.reshape(problem.rod.num_nodes, -1)[:, :num_trans]

### Solve

In [ ]:
u_star, u_history = problem.solve(
  u0=jnp.zeros(rod.num_dofs),
  shear_moduli=jnp.full((num_elems, num_fibers), shear_modulus),
)
history = np.asarray(u_history).reshape(num_steps, num_nodes, -1)

contact_forces = np.asarray(
  jax.vmap(lambda u, k: compute_contact_forces(problem, u, k))(
    jnp.asarray(u_history), jnp.arange(num_steps)
  )
)
force_magnitude = np.linalg.norm(contact_forces, axis=-1)
print(f"peak nodal contact force = {force_magnitude.max():.4f} N")
print(f"max nodal rotation       = "
      f"{np.degrees(np.abs(history[..., 3:]).max()):.1f} deg   (2 pi wall at 360)")

### Contact forces

In [ ]:
node_arc = np.asarray(rod.arc_coords)
fig, (ax_dist, ax_total) = plt.subplots(1, 2, figsize=(12, 4.5))
for step in range(0, num_steps, max(1, num_steps // 6)):
  ax_dist.plot(node_arc, force_magnitude[step], lw=1.2,
               label=f"mouth {mouth_positions[step]:.3f}")
ax_dist.set_xlabel(r"reference arc coordinate $\hat{s}$ [m]")
ax_dist.set_ylabel("nodal contact force [N]")
ax_dist.legend(fontsize=7)

ax_total.plot(mouth_positions, force_magnitude.sum(axis=1), "o-", ms=3, label="total")
ax_total.plot(mouth_positions, force_magnitude.max(axis=1), "s-", ms=3, label="peak")
ax_total.invert_xaxis()
ax_total.set_xlabel("pipe mouth position [m]")
ax_total.set_ylabel("contact force [N]")
ax_total.legend()
fig.tight_layout()
plt.show()

### Shapes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for step in np.linspace(0, num_steps - 1, 6, dtype=int):
  shape = np.asarray(rod.node_coords) + history[step, :, :3]
  ax.plot(shape[:, 0], shape[:, 1], lw=1.4)
ax.set_aspect("equal")
ax.set_xlabel("$x$ [m]")
ax.set_ylabel("$y$ [m]")
plt.show()

### Polyscope

In [ ]:
final_step = num_steps - 1
coords, directors, fiber_coords = (
  _mesher.extract_coords_and_director_from_deformation(
    rod, jnp.asarray(history[final_step].ravel())
  )
)
# Nodal forces averaged onto elements, which is what a curve network colours by.
elem_force = 0.5 * (
  force_magnitude[final_step, :-1] + force_magnitude[final_step, 1:]
)
node_radii = np.empty(num_nodes)
node_radii[0], node_radii[-1] = contact_radii[0], contact_radii[-1]
node_radii[1:-1] = 0.5 * (contact_radii[:-1] + contact_radii[1:])

_viz.init_scene_ps()
_viz.plot_rod_centerline_ps(
  np.asarray(coords),
  node_radii=node_radii,
  elem_field=elem_force,
  elem_field_val_range=(0.0, float(force_magnitude.max())),
  cmap="viridis",
  name="rod",
)
_viz.plot_obstacle_ps(obstacle, step_idx=final_step, name="pipe")
ps.show()

### Verification

Where the rod is fully seated against the inner wall of the fillet it has no choice
but to follow it, so its curvature equals the fillet curvature. The straight runs
force zero curvature. Both are exact geometric statements, independent of the
material, so any deviation is solver error rather than modelling error.

In [ ]:
final_disp = jnp.asarray(history[-1].ravel())
coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
  rod, final_disp
)
achieved = np.asarray(
  _mesher.compute_discrete_cosserat_kinematics(
    node_coords=coords, directors=directors, arc_coords=rod.arc_coords
  ).curvature
)
# Arc position of each element centre measured ALONG the pipe, from its mouth.
elem_arc = 0.5 * (node_arc[:-1] + node_arc[1:])
depth_into_pipe = elem_arc - mouth_positions[-1]
in_fillet = (depth_into_pipe > inlet_length) & (
  depth_into_pipe < inlet_length + fillet_radius * np.pi / 2.0
)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(elem_arc, achieved[:, 1], lw=1.5, label=r"$\kappa_2$ achieved")
ax.axhline(1.0 / fillet_radius, color="k", ls="--", lw=1.2,
           label=r"$1/R_{fillet}$")
ax.axhline(0.0, color="k", lw=0.5)
ax.axvspan(
  elem_arc[in_fillet].min(), elem_arc[in_fillet].max(), color="0.9", zorder=0
)
ax.set_xlabel(r"reference arc coordinate $\hat{s}$ [m]")
ax.set_ylabel(r"$\kappa_2$ [1/m]")
ax.legend()
plt.show()

print(f"mean kappa in the fillet = {achieved[in_fillet, 1].mean():.4f} 1/m")
print(f"fillet curvature 1/R     = {1.0 / fillet_radius:.4f} 1/m")
print(f"relative error           = "
      f"{abs(achieved[in_fillet, 1].mean() - 1.0 / fillet_radius) * fillet_radius:.3%}")

In [ ]:
import dataclasses

# Registering the obstacle with its schedule stripped draws it at the identity pose,
# which is its body frame. Everything else is then transformed into that frame, so
# the pipe is stationary and the rod is what moves.
static_pipe = dataclasses.replace(obstacle, movement=None)

force_range = (0.0, float(force_magnitude.max()))
node_radii = np.empty(num_nodes)
node_radii[0], node_radii[-1] = contact_radii[0], contact_radii[-1]
node_radii[1:-1] = 0.5 * (contact_radii[:-1] + contact_radii[1:])

_viz.init_scene_ps()
_viz.plot_obstacle_ps(static_pipe, step_idx=0, name="pipe")
_viz.plot_rod_centerline_ps(
  np.zeros((num_nodes, 3)),
  node_radii=node_radii,
  elem_field=np.zeros(num_elems),
  elem_field_val_range=force_range,
  cmap="viridis",
  name="rod",
)

# Fixed once, so Polyscope does not re-fit the view as the rod sweeps through.
ps.look_at(camera_location=(0.55, 0.35, 0.55), target=(0.12, 0.10, 0.0))

In [ ]:
import imageio.v2 as imageio
import pathlib

frame_dir = pathlib.Path("frames")
frame_dir.mkdir(exist_ok=True)

for step in range(num_steps):
  coords, _, _ = _mesher.extract_coords_and_director_from_deformation(
    rod, jnp.asarray(history[step].ravel())
  )
  obj_trans, obj_rot_mat = obstacle.pose_at_step(step)
  # Lab frame to pipe body frame. The pipe is drawn fixed, so the rod carries the
  # relative motion and the animation reads as an insertion.
  coords_in_pipe = _utils.affine_transform(
    coords, obj_trans, obj_rot_mat, "GLOBAL_TO_LOCAL"
  )
  elem_force = 0.5 * (force_magnitude[step, :-1] + force_magnitude[step, 1:])

  _viz.update_rod_centerline_ps(
    np.asarray(coords_in_pipe), elem_field=elem_force, name="rod"
  )
  ps.screenshot(str(frame_dir / f"frame_{step:03d}.png"), transparent_bg=False)

imageio.mimsave(
  "insertion.gif",
  [imageio.imread(frame_dir / f"frame_{step:03d}.png") for step in range(num_steps)],
)

In [ ]:
imageio.mimsave(
  "insertion.gif",
  [imageio.imread(frame_dir / f"frame_{step:03d}.png") for step in range(num_steps)],
)